In [69]:
import numpy as np
from graphblas import Matrix, Vector, semiring
from graphblas import binary, monoid
from graphblas.semiring import plus_times
import graphblas as gb
import time

# Задача 1 
Используя `python-graphblas` реализовать наивный алгоритм, для матрицы смежности $A$ вычисляющий $A^3$ и возвращающий количество треугольников неориентированного графа. 
- Функция принимает представление неориентированного графа, удобное для неё (загрузка, конвертация и проверка неориентированности реализованы отдельно). 
- Функция возвращает число --- количество треугольников в графе.

#### **Решение**

Если $A$ — матрица смежности неориентированного графа, то элемент $(A^3)[i][i]$ (диагональный элемент куба матрицы) равен удвоенному числу замкнутых путей длины 3, проходящих через вершину $i$.

Почему след $A^3$ связан с треугольниками:

- $(A^3)[i][j]$ = число путей длины 3 из $i$ в $j$.
- $(A^3)[i][i]$ = число замкнутых путей длины 3 из $i$ обратно в $i$.
- Каждый треугольник $(i, j, k)$ порождает для вершины $i$ ровно 2 замкнутых пути: $i \to j \to k \to i$ и $i \to k \to j \to i$.
- Каждый треугольник содержит 3 вершины, и на каждой — 2 пути.
- Итого: $\text{trace}(A^3) = 6 \times \text{число треугольников}$.

**Формула:**

$$\text{triangles} = \frac{\text{trace}(A^3)}{6}$$


In [70]:
def triangle_A3(A):

    # A² = A @ A
    A2 = A.mxm(A, semiring.plus_times)

    # A³ = A² @ A
    A3 = A2.mxm(A, semiring.plus_times)

    # находим trace
    trace = A3.diag().reduce().value

    return trace // 6

In [71]:
# Граф-треугольник: 0-1, 1-2, 0-2
A = Matrix.from_coo(
    [0, 0, 1, 1, 2, 2],
    [1, 2, 0, 2, 0, 1],
    [1, 1, 1, 1, 1, 1],
    nrows=3, ncols=3
)

# Граф с 2 треугольниками: (0,1,2) и (1,2,3)
# Рёбра: 0-1, 0-2, 1-2, 1-3, 2-3
B = Matrix.from_coo(
    [0, 0, 1, 1, 1, 2, 2, 2, 3, 3],
    [1, 2, 0, 2, 3, 0, 1, 3, 1, 2],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    nrows=4, ncols=4
)

print('Количество треугольников в графе А:',triangle_A3(A))
print('Количество треугольников в графе В:',triangle_A3(B))

Количество треугольников в графе А: 1
Количество треугольников в графе В: 2


# Задача 2
Используя `python-graphblas` реализовать наивный алгоритм, для матрицы смежности $A$ вычисляющий $A^2$ и возвращающий количество треугольников неориентированного графа. 
- Функция принимает представление неориентированного графа, удобное для неё (загрузка, конвертация и проверка неориентированности реализованы отдельно). 
- Функция возвращает число --- количество треугольников в графе.

#### **Решение**

**Идея** — оптимизация алгоритма через $A^3$, вместо того чтобы вычислять полный куб матрицы:

- $A^2[i][j]$ = число путей длины 2 из $i$ в $j$
- Если при этом существует ребро $(i, j)$ в графе (т.е. $A[i][j] \neq 0$), то каждый такой путь длины 2 вместе с этим ребром образует треугольник
- Значит, нам не нужен полный $A^3$ — достаточно вычислить $A^2$ и посмотреть только на те позиции, где уже есть ребро в $A$

При вычислении $A^2$ мы применяем матрицу $A$ как маску, оставляя только элементы $A^2[i][j]$, для которых $A[i][j]$ ненулевой.

**Формула:**

$$\text{triangles} = \frac{\sum_{(i,j)} (A^2 \circ A)[i][j]}{6}$$

где $\circ$ — поэлементное пересечение 

Делим на 6 по той же причине: каждый треугольник $(i,j,k)$ порождает 6 пар $(i,j)$ с путём длины 2 через третью вершину (3 ребра × 2 направления).


**Преимущество перед $A^3$**

Мы не вычисляем полное произведение $A^2 \times A$. Вместо этого:
- Вычисляем $A^2$
- Применяем маску $A$ 
- Суммируем результат

In [72]:
def triangle_A2(A):
    
    # Вычисляем A² = A @ A, но с маской A:
    # результат содержит A²[i][j] только там, где A[i][j] != 0.
    A2 = A.mxm(A, semiring.plus_times).new(mask=A.S)
    
    total = A2.reduce_scalar(monoid.plus).new().value

    return total // 6


print('Количество треугольников в графе А:',triangle_A2(A))
print('Количество треугольников в графе В:',triangle_A2(B))

Количество треугольников в графе А: 1
Количество треугольников в графе В: 2


# Задача 3
Используя `python-graphblas` реализовать `Сohen's algorithm`, вычисляющий количество треугольников неориентированного графа.
 
- Функция принимает представление неориентированного графа, удобное для неё (загрузка, конвертация и проверка неориентированности реализованы отдельно).
- Функция возвращает число --- количество треугольников в графе.

#### **Решение**

Алгоритм Cohen основан на приведении задачи подсчёта треугольников к операциям линейной алгебры над разреженными матрицами с предварительной ориентацией рёбер графа

Пусть дан неориентированный граф $G = (V, E)$ с матрицей смежности $A$. Требуется найти количество треугольников

**Ориентация рёбер**

Для каждого ребра $(u, v) \in E$ определяется направление по следующему правилу:

$$u \to v \iff \deg(u) < \deg(v) \lor (\deg(u) = \deg(v) \land u < v)$$

Результатом является ориентированный ациклический граф (DAG) с матрицей смежности $L$, где каждое неориентированное ребро представлено ровно одной направленной дугой

#### Подсчёт треугольников

1. Вычисляется матрица $B = L \cdot L$ . Элемент $B[u][w]$ равен числу вершин $v$ таких, что в DAG существуют дуги $u \to v$ и $v \to w$, то есть числу путей длины 2 из $u$ в $w$.

2. Выполняется поэлементное произведение $C = L \circ B$. Ненулевой элемент $C[u][w]$ означает, что одновременно существуют дуга $u \to w$ в $L$ и хотя бы один путь длины 2 из $u$ в $w$ через промежуточную вершину $v$. Тройка $(u, v, w)$ образует треугольник.

3. Количество треугольников вычисляется как сумма всех элементов матрицы $C$:

$$|\Delta| = \sum_{u,w} C[u][w]$$

Деление не требуется, поскольку ориентация по степеням гарантирует, что каждый треугольник обнаруживается ровно один раз: для любого треугольника $(a, b, c)$ существует единственная упорядоченная тройка, в которой дуги $u \to v$, $v \to w$ и $u \to w$ принадлежат $L$.

In [73]:
def count_triangles_sandia(A: gb.Matrix) -> int:

    # берём только верхнюю треугольную часть
    U = gb.select.triu(A)

    # пересечение соседей
    M = U.mxm(U.T)

    # маска по исходной матрице
    M = M.ewise_mult(U)

    # суммирование элементов
    total = M.reduce_scalar(gb.agg.sum).value

    if total is None:
        total = 0

    return int(total)

In [74]:
print('Количество треугольников в графе А:',count_triangles_sandia(A))
print('Количество треугольников в графе В:',count_triangles_sandia(B))

Количество треугольников в графе А: 1
Количество треугольников в графе В: 2


# Задача 4
Используя `python-graphblas` реализовать `Sandia algorithm`, вычисляющий количество треугольников неориентированного графа.
- Функция принимает представление неориентированного графа, удобное для неё (загрузка, конвертация и проверка неориентированности реализованы отдельно).
- Функция возвращает число --- количество треугольников в графе.

#### **Решение**

Алгоритм Sandia представляет собой метод подсчёта треугольников, основанный на вычислении нижнетреугольной части матрицы смежности.

Пусть дан неориентированный граф $G = (V, E)$ с симметричной матрицей смежности $A$. Требуется найти количество треугольников.

Из симметричной матрицы $A$ извлекается строго нижнетреугольная часть $L = \text{tril}(A)$, содержащая только элементы $A[i][j]$ при $i > j$. 

Это эквивалентно ориентации всех рёбер от вершины с большим индексом к вершине с меньшим: каждое неориентированное ребро $(u, v)$ представляется ровно одной записью в $L$.

**Подсчёт треугольников**

1. Вычисляется произведение $B = L \cdot L$

2. Применяется маска $L$: $C = L \circ B$. Ненулевой элемент $C[i][j]$ означает, что существует дуга $i \to j$ в $L$ и одновременно хотя бы один путь длины 2 из $i$ в $j$, что соответствует наличию треугольника.

3. Количество треугольников вычисляется как сумма всех элементов $C$:

$$|\Delta| = \sum_{i,j} C[i][j]$$

Деление не требуется. Поскольку $L$ — строго нижнетреугольная матрица, для каждого треугольника существует единственная тройка дуг, при которой этот треугольник обнаруживается ровно один раз.

**Отличие от алгоритма Коэна**

В алгоритме Коэна ориентация рёбер выполняется по правилу $(\deg(u), u)$, что ограничивает максимальную исходящую степень величиной $O(\sqrt{m})$ и обеспечивает сложность $O(m^{3/2})$

Алгоритм Sandia использует ориентацию строго по индексу вершины (нижнетреугольная часть), что проще в реализации, однако не даёт гарантии на ограничение исходящей степени

In [75]:
def sandia_triangle(A):

    # извлечение строго нижнетреугольной части
    L = A.select("tril", -1).new()

    B = L.mxm(L, semiring.plus_times).new(mask=L.S)

    return B.reduce_scalar(monoid.plus).new().value

In [76]:
print('Количество треугольников в графе А:',sandia_triangle(A))
print('Количество треугольников в графе В:',sandia_triangle(B))

Количество треугольников в графе А: 1
Количество треугольников в графе В: 2


# Задача 5
Используя `python-graphblas` реализовать функцию, вычисляющую для каждой вершины неориентированного графа количество треугольников, в которых она участвует.
- Функция принимает представление неориентированного графа, удобное для неё (загрузка, конвертация и проверка неориентированности реализованы отдельно).
- Функция возвращает массив, где для каждой вершины указано, в скольки треугольниках она участвует.

#### **Решение**

Пусть дан неориентированный граф $G = (V, E)$ с симметричной матрицей смежности $A$

Из наивного алгоритма с маской известно, что матрица $C = A^2 \circ A$ содержит информацию о треугольниках: элемент $C[i][j]$ равен числу вершин $k$, таких что $(i, k, j)$ образует треугольник, при условии что ребро $(i, j)$ существует.

Для получения числа треугольников на вершину выполняется приведение по строкам:

$$t(v) = \sum_{j} C[v][j]$$

Однако каждый треугольник $(v, j, k)$ при таком подсчёте учитывается дважды через вершину $v$: один раз при суммировании $C[v][j]$ (путь $v \to k \to j$) и один раз при суммировании $C[v][k]$ (путь $v \to j \to k$). Поэтому:

$$\delta(v) = \frac{t(v)}{2}$$

In [77]:
def vertex(A):

    A2 = A.mxm(A, semiring.plus_times).new()

    C = A2.ewise_mult(A).new()

    t = C.reduce_rowwise(monoid.plus).new()

    result = t.apply(gb.unary.identity).new()
    result << result // 2

    return result

### Извлечение результата в массив
def vertex_array(A):
    vec = vertex(A)
    n = A.nrows
    return [vec.get(i, 0) for i in range(n)]

In [78]:
print('Кол-во треугольников для каждой вершины в графе А (1 треугольник):',vertex_array(A))
print('Кол-во треугольников для каждой вершины в графе В (2 треугольника):',vertex_array(B))

Кол-во треугольников для каждой вершины в графе А (1 треугольник): [1, 1, 1]
Кол-во треугольников для каждой вершины в графе В (2 треугольника): [1, 2, 2, 1]


# Задача 6
Используя `python-graphblas` реализовать функцию, вычисляющую для каждой вершины неориентированного графа количество треугольников, в которых она участвует.
- Функция принимает представление неориентированного графа, удобное для неё (загрузка, конвертация и проверка неориентированности реализованы отдельно).
- Функция возвращает массив, где для каждой вершины указано, в скольки треугольниках она участвует.

#### **Решение**

In [79]:
import glob
import os

folder_path = r"C:\Projects\ITMO_graph\Matrix"

mtx_files = glob.glob(folder_path + "/*.mtx")

graphs = []

for file in mtx_files:
    print("Загрузка:", file)
    A = gb.io.mmread(file)

    filename = os.path.basename(file)

    graphs.append((filename, A))

print(len(graphs))

Загрузка: C:\Projects\ITMO_graph\Matrix\bcspwr10.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\bcsstk29.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\bcsstk32.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\bcsstk33.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\can_838.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\dwt_992.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\eris1176.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\lshp2233.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\lshp3025.mtx
Загрузка: C:\Projects\ITMO_graph\Matrix\lshp3466.mtx
10


In [80]:
algorithms = {
    "Naive": triangle_A3,
    "Mask": triangle_A2,
    "Cohen": count_triangles_sandia,
    "Sandia": sandia_triangle,
    "PerVertex": vertex
}

In [81]:
import pandas as pd

results = []

for name, A in graphs:

    row = {"Graph": name}

    for alg_name, alg_func in algorithms.items():

        times = []

        for _ in range(3):  # усредняем
            start = time.time()
            alg_func(A)
            times.append(time.time() - start)

        row[alg_name] = sum(times) / len(times)

    results.append(row)


df = pd.DataFrame(results)
df

,Graph,Naive,Mask,Cohen,Sandia,PerVertex
0,bcspwr10.mtx,0.004665,0.001333,0.001667,0.000333,0.002667
1,bcsstk29.mtx,0.026699,0.006333,0.007667,0.004333,0.009667
2,bcsstk32.mtx,0.082350,0.017666,0.018667,0.009000,0.056043
3,bcsstk33.mtx,0.035666,0.007663,0.008334,0.004332,0.012541
4,can_838.mtx,0.001999,0.000333,0.000669,0.000000,0.000997
5,dwt_992.mtx,0.001667,0.000333,0.001002,0.000232,0.001040
6,eris1176.mtx,0.002079,0.000546,0.000668,0.000333,0.001334
7,lshp2233.mtx,0.002333,0.000669,0.000664,0.000000,0.001336
8,lshp3025.mtx,0.003000,0.000664,0.000667,0.000336,0.015343
9,lshp3466.mtx,0.003664,0.000667,0.000669,0.000000,0.001667


### Вывод

1. Sandia — безусловный лидер на всех графах

2. Mask(A²) — Второй результат по скорости, в 4–5 раз быстрее Naive. Одно умножение вместо двух и маска позволяют увеличить скорость

3. Cohen — Показывает почти такие же результаты как и Mask 

4. PerVertex работает медленнее остальных, но это логично, потоу что он считает более общую задачу, ищет кол-во треугольников на каждую вершину

5. Naive(A³) - Во всех тестах наихудшее время работы показывает алгоритм Naive. Он стабильно работает медленнее остальных методов
